## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from pathlib import Path
from src.paths import DATA_PROCESSED


## 2. Chargement des données

In [2]:
df = pd.read_csv(DATA_PROCESSED / "movies_preprocessed_clean.csv")
cast = pd.read_csv(DATA_PROCESSED / "df_cast.csv")

Je vais essayer d'enrichire le dataset avec du casting, 
Je pense garder seulement les noms des acteurs/actrices et directeurs seulement pour le moment

In [3]:
roles = ['actor', 'actress', 'director']
cast = cast[cast['category'].isin(roles)]

In [4]:
print(cast.shape)
cast.head()

(4798670, 5)


,tconst,nconst,category,primaryName,knownForTitles
0,tt0000009,nm0063086,actress,Blanche Bayliss,tt0000009
1,tt0000009,nm0183823,actor,William Courtenay,"tt0000009,tt0020355,tt0021535,tt0020403"
2,tt0000009,nm1309758,actor,Chauncey Depew,"tt0000009,tt0490842,tt1076833,tt4484306"
3,tt0000009,nm0085156,director,Alexander Black,tt0000009
12,tt0000147,nm0714557,director,Enoch J. Rector,"tt0381108,tt0000147,tt0229676"


In [5]:
cast['category'].unique()

<StringArray>
['actress', 'actor', 'director']
Length: 3, dtype: str

Ok on passe de 8551946 lignes a 4798670 après le tri, nous avons bien uniquement actress actor et director 
Afin de conserver les noms complet je vais supprimer les espaces entres nom/prenoms et tout mettre en minuscule

In [6]:
cast['clean_name'] = cast['primaryName'].str.lower()
cast['clean_name'] = cast['clean_name'].str.replace(' ', '', regex=False)
cast['clean_name'] = cast['clean_name'].str.replace('.', '', regex=False)

Suite a une erreur plus bas je verifie les NA , effectivement 211 NA donc on les supprime , un acteur sans nom sert a rien 


In [7]:
cast['clean_name'].isna().sum()

np.int64(211)

In [8]:
cast = cast.dropna(subset=['clean_name'])

In [9]:
cast.head()



,tconst,nconst,category,primaryName,knownForTitles,clean_name
0,tt0000009,nm0063086,actress,Blanche Bayliss,tt0000009,blanchebayliss
1,tt0000009,nm0183823,actor,William Courtenay,"tt0000009,tt0020355,tt0021535,tt0020403",williamcourtenay
2,tt0000009,nm1309758,actor,Chauncey Depew,"tt0000009,tt0490842,tt1076833,tt4484306",chaunceydepew
3,tt0000009,nm0085156,director,Alexander Black,tt0000009,alexanderblack
12,tt0000147,nm0714557,director,Enoch J. Rector,"tt0381108,tt0000147,tt0229676",enochjrector


C'est propre et beaucoup plus lisible pour un algorithme , je vais regrouper les films avec acteurs/actrices en une seule ligne baser sur le tconst

In [10]:
cast_group = cast.groupby('tconst')['clean_name'].agg(' '.join).reset_index()

In [11]:
cast_group.head()

,tconst,clean_name
0,tt0000009,blanchebayliss williamcourtenay chaunceydepew ...
1,tt0000147,enochjrector
2,tt0000502,antoniodelpozo elmochuelo ricardodebaños
3,tt0000574,elizabethtait johntait nicholasbrierley norman...
4,tt0000591,georgeswague henrigouget christianemandelys gi...


Je sauvegarde le fichier propre et nettoyer 

In [12]:
cast_group.to_csv(DATA_PROCESSED / "cast_group_clean.csv")